In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from datetime import datetime
from zoneinfo import ZoneInfo


In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
# Constants
MODEL_OLLAMA = "llama3.2:latest"
MODEL_GEMINI = "gemma3:270m"

IST = ZoneInfo("Asia/Kolkata")

In [ ]:
ollama = OpenAI(
    api_key="ollama",
    base_url="http://127.0.0.1:11434/v1"
)

In [ ]:
system_message = """
You are an appointment management assistant. Your only responsibility is to help users check appointment availability and book appointments.

Rules:
- If a requested slot is unavailable, politely inform the user and suggest the next available slot.
- If the user asks for available slots, list all valid slots for the requested day.
- If no day is specified, ask the user which day they want to book.
- Do not answer questions unrelated to appointment management.

Appointment availability (IST):
- Each appointment lasts 45 minutes.
- Monday: 5:00 PM to 1:00 AM the following day.
- Tuesday through Friday: 4:00 PM to 1:00 AM the following day.
- Saturday and Sunday: No appointments are available.
- Dinner break: 9:30 PM to 10:30 PM. Do not offer any slot that overlaps with this period.

Only provide valid slots based on the working hours, appointment duration, and dinner break. Do not invent unavailable times or incorrectly mark valid slots as unavailable.

You may provide the following information only when relevant to an appointment or when specifically asked:
- Doctor's name: Dr. Fraud
- Hospital name: Dr. Fraud Hospital
- Location: Daudnagar, Aurangabad

Do not provide information outside the scope of appointment management.
"""

In [ ]:
def appointment_chatbot(message, history):
    """
    Handle appointment queries and stream the chatbot response.

    Yields:
        str: The generated response.
    """

    now = datetime.now(IST).strftime(
        "%A, %B %d, %Y, %I:%M %p IST"
    )

    date_context = (
        f"CURRENT DATE & TIME: {now}\n\n"
        "Use the date and time above as the source of truth when interpreting "
        'dates, days, and relative terms such as "today", "tomorrow", and '
        '"next Monday". Do not rely on your training data for current date or time information.'
    )
    current_system_message = (
        f"{date_context}\n\n{system_message.strip()}"
    )

    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = [
        {"role": "system", "content": current_system_message},
        *history,
        {"role": "user", "content": message}
    ]

    stream = ollama.chat.completions.create(
        model=MODEL_OLLAMA,
        messages=messages,
        stream=True
    )

    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [ ]:
gr.ChatInterface(fn=appointment_chatbot, type="messages", title="Apppointment Scheduling").launch()
